# Bind inputs and time schedules

[Write a calculation](write-a-calculation.ipynb) feeds one block its heating through `funcs`, as a constant and then as a step. This page uses the same mechanism on a loop of the library's components: it binds a pump's head as a constant, as a step and as a table in time, states what a binding cannot reach, and sweeps the head over five values on one aggregator.

A binding is `funcs={calc: {"name": value}}`, and `FlowGraph`, `CalculationGraph`, `Aggregator` and `from_decoupled` all take it. The value is a constant or a callable of the time alone; the aggregator calls it at every evaluation of $F$ and hands the result to `calc` as the keyword `name`.

In [1]:
import numpy as np
import pandas as pd

from stream.analysis.report import report
from stream.calculations import HeatExchanger, Junction, Pump, Resistor
from stream.composition import FlowGraph, flow_edge

## A constant

The loop is a pump, a core resistance of `2.5e4` Pa per kg/s and a heat exchanger returning the water at 40 °C, on three named junctions. A `Pump` takes a bound `pressure` as its head, ahead of any head it was built with; this one is built with none. The loop has no change of height, so its flow is the head over the resistance, $\dot m = H / R$.

In [2]:
j_pump, j_core, j_hx = Junction(name="J_pump"), Junction(name="J_core"), Junction(name="J_hx")
pump = Pump(name="Pump")
core = Resistor(2.5e4, name="Core")
hx = HeatExchanger(outlet=40.0, name="HX")

fg = FlowGraph(
    flow_edge((j_pump, j_core), pump),
    flow_edge((j_core, j_hx), core),
    flow_edge((j_hx, j_pump), hx),
    funcs={pump: {"pressure": 5e3}},
    reference_node=(j_pump, 1e5),
)
agr, K = fg.aggregator, fg.kirchhoff
report(agr)

y = agr.solve_steady(fg.guess_steady_state({pump: 0.1, core: 0.1, hx: 0.1}, 40.0))
print(f"flow solved:  {agr.save(y)[K.name][K.component_edge(core)]:.4f} kg/s")
print(f"flow by hand: {5e3 / core.r:.4f} kg/s")

Contains 12 Equations, 7 Nodes, 21 Edges.

| Calculation | Type | Equation No. | Unset | Set Externally | Missing |
| :---------- | :--- | :----------- | :---- | :------------- | :------ |
| J_pump | Junction | 0 - 1 |  |  |  |
| Pump | Pump | 1 - 3 | mdot0 | pressure |  |
| J_core | Junction | 3 - 4 |  |  |  |
| Core | Resistor | 4 - 6 |  |  |  |
| J_hx | Junction | 6 - 7 |  |  |  |
| HX | HeatExchanger | 7 - 9 |  |  |  |
| Kirchhoff | Kirchhoff | 9 - 12 |  |  |  |


flow solved:  0.2000 kg/s
flow by hand: 0.2000 kg/s


The pump's `pressure` stands under **Set Externally**, and the flow is the hand value, about 0.2 kg/s.

## A step and a table in time

`solve_steady` evaluates every binding at time zero, so a schedule shows only in a solve over time. This loop has no differential row, and `solve` finds its root at each output time from the root at the one before, a quasi-static solve. The step drops the head to half at 10 s. The table is `np.interp` over a few points, which holds the end values outside them: the head holds until 10 s and ramps linearly to zero by 20 s. Both are rebound on the same aggregator, which the last section explains.

The clock itself is bound the same way: a calculation whose `calculate` takes `t` gets it from `{"t": identity}`, with `identity` from `stream.utilities` returning its argument.

In [3]:
schedules = {
    "step": lambda t: 5e3 if t < 10.0 else 2.5e3,
    "table": lambda t: np.interp(t, [0.0, 10.0, 20.0], [5e3, 5e3, 0.0]),
}
time = np.linspace(0.0, 20.0, 5)
flows = {}
for label, head in schedules.items():
    agr.funcs[pump]["pressure"] = head
    sol = agr.solve(y, time)
    flows[f"{label} [kg/s]"] = agr.at_times(sol, K, K.component_edge(core))
    flows[f"{label} by hand [kg/s]"] = [head(t) / core.r for t in time]
pd.DataFrame(flows, index=pd.Index(time, name="time [s]")).round(4)

,step [kg/s],step by hand [kg/s],table [kg/s],table by hand [kg/s]
time [s],,,,
0.0,0.2,0.2,0.2,0.2
5.0,0.2,0.2,0.2,0.2
10.0,0.1,0.1,0.2,0.2
15.0,0.1,0.1,0.1,0.1
20.0,0.1,0.1,0.0,0.0


Each column follows its schedule at every output time. A schedule runs at every evaluation of $F$, not once per output time: a cheap function of the time alone, giving the same value each time it is asked, is the right size.

## What a binding cannot reach

**The state.** A binding is passed the time and nothing else, though a closure may read an attribute that an event sets, as the decay-heat schedule of the pool reactor does. A driver that watches the state, a trip reading a flow, is a `Calculation` with rows and events of its own, as `LowFlowTrip` in `stream.examples.lofa.components` is; [Trip, stop and restart](trip-stop-and-restart.ipynb) builds one.

**Another object.** `funcs` is keyed by the calculation object, as [Write a calculation](write-a-calculation.ipynb) shows, so a copy of a calculation is another key and needs its own entry.

**An edge's value.** A binding on a name an edge already supplies replaces the edge's value, with no warning. **Set Externally** lists it, which is where to catch it.

**An attribute.** A value the component keeps on itself is not an input. `HeatExchanger` holds its outlet temperature as `hx.T`; a binding named `outlet` reaches its `calculate` and is ignored, and `report` does not list it. Changing such a value is changing the component, as [Primary Cooling System Coastdown](../PCSCoastdown.ipynb) does to its pump to start a transient.

In [4]:
agr.funcs[hx] = {"outlet": 60.0}
report(agr)
del agr.funcs[hx]

Contains 12 Equations, 7 Nodes, 21 Edges.

| Calculation | Type | Equation No. | Unset | Set Externally | Missing |
| :---------- | :--- | :----------- | :---- | :------------- | :------ |
| J_pump | Junction | 0 - 1 |  |  |  |
| Pump | Pump | 1 - 3 | mdot0 | pressure |  |
| J_core | Junction | 3 - 4 |  |  |  |
| Core | Resistor | 4 - 6 |  |  |  |
| J_hx | Junction | 6 - 7 |  |  |  |
| HX | HeatExchanger | 7 - 9 |  |  |  |
| Kirchhoff | Kirchhoff | 9 - 12 |  |  |  |


The heat exchanger's row has nothing under **Set Externally**.

## Sweep a boundary condition

The aggregator reads `agr.funcs` at every evaluation, and `report` reads it when called, so a binding changed on a built aggregator needs no rebuild. Rebind through `agr.funcs`: once aggregators are added, the sum holds a merged copy of their bindings, not the dictionaries passed at construction. The cell sweeps the head over five values, each solve starting from the last.

In [5]:
rows = []
for head in [2.5e3, 5e3, 7.5e3, 1e4, 1.25e4]:
    agr.funcs[pump]["pressure"] = head
    y = agr.solve_steady(y)
    rows.append({"head [kPa]": head / 1e3, "flow [kg/s]": agr.save(y)[K.name][K.component_edge(core)],
                 "H / R [kg/s]": head / core.r})
pd.DataFrame(rows).round(4)

,head [kPa],flow [kg/s],H / R [kg/s]
0,2.5,0.1,0.1
1,5.0,0.2,0.2
2,7.5,0.3,0.3
3,10.0,0.4,0.4
4,12.5,0.5,0.5


Every solved flow equals $H / R$ to the digits the table keeps.

[Building the Pool Reactor Loop](../LOFABuildingTheLoop.ipynb) binds at scale: a constant plate power, the clocks of the flapper and the trip, and a decay-heat schedule counted from the trip. The usual next task: [Add point kinetics, feedback and a trip](add-point-kinetics-and-a-trip.ipynb).